# Prefect secret blocks for `env-vars` and `env-vars-<login>`

* Jira: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-869
* ICD: https://pforge-exchange2.astrium.eads.net/confluence/pages/viewpage.action?pageId=495624025

## NOTE: manual action required on the cluster

We need to initialize the Prefect block that contains the environment variables for all users. This is not needed in local mode.

See: https://github.com/RS-PYTHON/rs-demo?tab=readme-ov-file#initialize-the-prefect-blocks

In [ ]:
# Set these values to False (default in the Airbus cluster) 
# so they don't appear in the prefect blocs
import os
os.environ["OTEL_PYTHON_REQUESTS_TRACE_HEADERS"] = "0"
os.environ["OTEL_PYTHON_REQUESTS_TRACE_BODY"] = "0"

In [ ]:
# Init the demos
from resources.utils import *
init_demo()

# Reload the global vars again
from resources.utils import *  

In [ ]:
# Imports
import json
import ipywidgets as widgets
from prefect import get_client
from prefect.blocks.system import Secret
from rs_client.osam_client import BucketCredentials
from rs_common.prefect_utils import *

hide = widgets.Checkbox(
    value=True,
    description="Hide environment variable values",
    indent=False,
)
display(hide)

In [ ]:
async def display_blocks():
    """Display prefect block values"""
    for block_name in "env-vars", f"env-vars-{OWNER_ID}":
        print(f"Contents of Prefect block: {block_name!r}:")
        for key, value in (await Secret.load(block_name)).get().items():
            print(f"  - {key}: {"***" if hide.value else value}")
        print()

In [ ]:
# Display default blocks
await display_blocks()

In [ ]:
# Save some extra bucket credentials
await save_bucket_credentials(
    osam_client,
    {
        "obs1": BucketCredentials(
            access_key="access1", 
            secret_key="secret1", 
            endpoint="endpoint1", 
            region="region1",
        ),
        "obs2": BucketCredentials(
            access_key="access2", 
            secret_key="secret2", 
            endpoint="endpoint2", 
            region="region2",
        ),
    }
)
await display_blocks()

In [ ]:
# Nothing changes the next time we call init_demo,
# the blocks are not overwritten.
init_demo()
await display_blocks()

In [ ]:
# We can remove specific bucket credentials
await remove_bucket_credentials("obs1")
await display_blocks()